# GEMS P7 — GeoFaultFormer (CNN + local Transformer + ConvGRU + structural heads + fault tracer) | Colab GPU

Controlled research ladder for fault **discovery**: T0 references (B0, B1, fusion, W2) → **T1** CNN + shifted-window attention (default; only rung trained by default) → T1_cnn matched pure-CNN control → T2 + centerline / orientation → T3 + ConvGRU → T4 endpoint tracer → T5 W2 + P7 features. Every stage is a repository function (`gems_bench/p7/run.py`); no modelling code lives here.

* **Independent**: needs no other notebook's memory. It reads (never rewrites) the M9 feature registry / cube cache, the frozen W2 maps and the boost configuration from `MyDrive/GEMS_2/outputs/`.
* **Writes** everything under `MyDrive/GEMS_2/outputs/p7_geofaultformer/` (`environment/`, `folds/`, `features/`, `experiments/<id>/{checkpoints,manifests,metrics,predictions,processed,figures,reports}`, `comparisons/`, `candidate_recall/`, `stacker/`, `selection/`, `submissions/`). Experiment ids carry the configuration fingerprint: nothing is overwritten.
* **Resumable**: after a disconnect run sections 0–3, then the section you were in. Training resumes from the last checkpoint of the same spec; a changed feature contract refuses to resume.
* **Nothing has been executed by the authoring session.** No P7 number exists until this notebook writes it. Docs: `docs/p7/P7_GEOFAULTFORMER_HANDOFF.md`.

## 0 — Configuration (edit before running)

Runtime → Change runtime type → **GPU** (L4 or A100; T4 works with micro-batch 8 × accumulation 2) and **High-RAM**. Pin the pushed commit in `CONFIG['ref']` and `EXPECT['commit']`. The default trains **T1 only**; enable the next rung only after reading the previous rung's comparison (section 8, 'Next decision').

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                      # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False,
    'train_baselines': [],
    'run_full_inference': False,
    'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': True,
    'seed': 2026,
}
PRIVATE_REPO = False
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}   # M9 is only READ (registry, cube cache)
P6_CONFIG = {'overwrite_existing': False}
BOOST_CONFIG = {'geology': {'backfill_missing': False}}          # boost = official scoring calls + references

P7_RUN = {
    'preflight': True,
    'prepare_features': True,
    'train_t1': True,
    'train_t1_cnn': False,             # matched pure-CNN control: needed to test H1 at a matched trainer
    'train_t2': False,
    'train_t3': False,
    'train_t4': False,                 # tracer on T2 held-out maps (needs T2 trained + evaluated)
    'evaluate': True,
    'candidate_recall': True,
    'w2_integration': False,           # T5: needs every unit's P7 held-out map
    'full_inference': False,
    'selection': False,
    'export': False,
    'figures': True,
    'report': True,
}

P7_CONFIG = {
    # 'scope': {'train_folds': 'all', 'train_splits': 'all'},   # subsets are smoke tests only (INCOMPLETE)
    # 'inputs': {'geological': {'enabled': False}},              # G02 full-grid channels (heavy, unmeasured)
    # 'data': {'patch_px': 256, 'augmentation': 'conservative'},
    # 'training': {'epochs': 20, 'steps_per_epoch': 100, 'effective_batch': 16},
}
P7_GPU = {
    'select': 'auto',                  # auto | T4 | L4 | A100 | CPU
    'max_runtime_min': 0,              # > 0: clean resumable stop at an epoch boundary
    'require_gpu': True,
}
EXPERIMENTS_TO_EVALUATE = None         # None = every rung trained under the current configuration
FULL_INFERENCE_EXPERIMENT = 'T1'

EXPECT = {'code_rev': 'p7.1-r1', 'commit': None}

## 1 — Google Drive, strict repository clone, version lock

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
    if not token:
        raise RuntimeError('PRIVATE_REPO=True but GITHUB_TOKEN is missing in Colab Secrets')
    hdr = base64.b64encode(('x-access-token:' + token).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'p7_colab_active', 'origin/' + ref)
else:                                   # a SHA / tag must resolve; never fall back to main
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime (stale module code)')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('Requested:', ref, '| resolved SHA:', HEAD)

## 2 — Start-up check + GPU detection (fail fast)

Training requires CUDA (`P7_GPU['require_gpu']`): it raises `GPURequired` instead of silently using the CPU. Evaluation-only sessions may set `require_gpu=False` (inference on CPU is slow).

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.p7 as P7PKG
from gems_bench.p6 import gpu_runtime as GR
if P7PKG.P7_CODE_REV != EXPECT['code_rev']:
    raise RuntimeError(f"code rev {P7PKG.P7_CODE_REV} != expected {EXPECT['code_rev']}: wrong commit checked out")
if EXPECT['commit'] and not HEAD.startswith(EXPECT['commit']):
    raise RuntimeError(f"HEAD {HEAD} != expected {EXPECT['commit']}")
GPU_INFO = GR.detect()
print(json.dumps(GPU_INFO, indent=2, default=str))
if P7_GPU.get('require_gpu', True) and any(P7_RUN[k] for k in ('train_t1', 'train_t1_cnn', 'train_t2', 'train_t3')):
    GR.require_gpu(GPU_INFO, {'required': True, 'min_free_gb': 6.0})
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi: no GPU visible')

## 3 — Attach data, M9 context, P6 references (read-only) and earlier P7 outputs

Stages cached Drive data (never rebuilt), the M9 feature registry, the frozen W2 maps (`p6_fast_track/stack`) and any earlier `p7_geofaultformer` outputs. Prints the status of every P7 stage found on Drive.

In [ ]:
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack
from gems_bench.p7 import P7GeoFaultFormer

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for step in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(step)                     # validate + stage cached Drive data, never rebuild
m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
p7 = P7GeoFaultFormer(p6, P7_CONFIG, gpu=P7_GPU, boost_config=BOOST_CONFIG)
print(json.dumps(p7.attach(), indent=2, default=str))
print('Output root:', p7.ws.root)

## 4 — Preflight: environment, channel contract, fold / leakage audits, memory profile

* **Inputs**: M9 cube channels (registry order, discovered at run time), optional extra layers (off by default). Every channel must be label-free and reproducible on the full grid (`features/channel_registry.csv`).
* **Audits**: `folds/leakage_overlap_audit.csv` (allowed ∩ inner / held-out zones, visible labels in held-out zones, withheld systems visible or weighted), `folds/fold_provenance.csv` (mask hashes).
* **Memory**: peak GPU memory and seconds per step of every trained rung on **synthetic** tensors (`environment/memory_profile.csv`). Planning estimates (unmeasured): T1 ≈ 10 GB at micro-batch 16 on L4, T3 ≈ 15 GB.
* **Blockers** printed here stop nothing automatically; read them before training.

In [ ]:
if P7_RUN['preflight']:
    PF = p7.preflight(profile_memory=True)
    print('Blockers:', PF['blockers'] or 'none')
    print('Modalities:', PF['env']['modalities'], '| channels:', PF['env']['n_channels'], '| preset:', PF['env']['preset'])
    display(PF['overlap'])
    display(PF['availability'])
    display(PF['memory'])

## 5 — Features: contract snapshot, target audit, label-free geographic shift

Builds / opens the M9 cube (memory-mapped), writes `features/feature_contract.json`, audits every unit's targets (`folds/target_audit.csv`: targets only from visible labels) and the per-channel shift between each geo unit's training geography and its held-out fold (`features/geographic_shift.csv`).

In [ ]:
if P7_RUN['prepare_features']:
    FE = p7.prepare_features()
    display(FE['target_audit'])
    print('Feature stage status:', FE['status'], FE.get('reason', ''))

## 6 — Training (GPU, resumable)

| rung | hypothesis | change vs parent |
|---|---|---|
| T1 | H1 multi-scale context | CNN + shifted-window attention at 1/8, 1/16; fault head (loss L0) |
| T1_cnn | H1 control | identical, attention bypassed |
| T2 | H2 structural supervision | + centerline + axial orientation heads (L2) |
| T3 | H3 recurrent refinement | T2 + ConvGRU (3 shared steps at 1/4) |

* Units: 3 geographic folds (`geo_f<k>`: checkpoint on the inner-validation zone, status per epoch VALID / INVALID_NUMERIC / NO_ELIGIBLE_VALIDATION) then 3 discovery splits (`disc_s<d>`: inner status NOT_CONFIGURED by design; fixed epochs = median best epoch of the geo units).
* Outputs: `experiments/<id>/checkpoints/<unit>_{model,last,best,latest}.pt`, `<unit>_selection.json`, `metrics/history_<unit>.csv` (losses, lr, inner score + status, GPU peak MB, samples/s), `manifests/{experiment.json, units.csv}`.
* Runtime (unmeasured planning figure): T1 20 epochs × 100 steps per unit, 6 units; read the first epochs' `epoch_s` and set `max_runtime_min` for a free-tier session.
* Possible blockers: `RuntimeError: no admissible patch` (zone too fragmented for the patch size → lower `data.patch_px`), CUDA OOM beyond the automatic halving (choose a smaller preset).

In [ ]:
TRAINED = {}
for flag, rung in (('train_t1', 'T1'), ('train_t1_cnn', 'T1_cnn'), ('train_t2', 'T2'), ('train_t3', 'T3')):
    if P7_RUN[flag]:
        out = p7.train(rung)
        TRAINED[rung] = out
        if out is None:
            print(rung, 'INTERRUPTED (runtime budget): re-run this cell to resume')
            continue
        print(rung, out['status'], out['experiment_id'], '| disc epochs:', out['disc_epochs'])
        display(out['units'][['unit', 'kind', 'status', 'selection', 'best_epoch', 'best_inner', 'epochs_run', 'n_params', 'seconds', 'reused']])
display(p7.registry())
display(GR.memory_report())

## 7 — Official evaluation (nested, paired with B0 / B1 / fusion / W2)

Held-out maps of every unit → rank inside the unit's evaluation mask → `Boost.eval_score` (official TP / FP / FN, eligible budget, known faults masked, system hits) for every (rep, frac) cell → `export_parity.evaluate_policy` (one cell per outer zone chosen on the other zones). References are scored by the same calls on the same units.

Outputs: `comparisons/{summary, per_zone, paired_zone_deltas, matched_cell_deltas, hypotheses}.csv`, `experiments/<id>/metrics/{grid, summary, per_zone, prediction_mass}.csv`, `experiments/<id>/predictions/{oof_*.tif, disc_*.tif, units/*.npz}`.

Keep apart: inner checkpoint scores (training logs), nested outer DTI (here), matched-cell deltas (diagnostic), public leaderboard (never computed here).

In [ ]:
if P7_RUN['evaluate']:
    EVAL = p7.evaluate(EXPERIMENTS_TO_EVALUATE)
    cols = [c for c in ('policy', 'status', 'disc_dti', 'disc_worst', 'geo_dti', 'geo_worst', 'system_recall', 'disc_fp_w', 'disc_coverage_w', 'deploy_cell') if c in EVAL['summary']]
    display(EVAL['summary'][cols])
    display(EVAL['hypotheses'])

## 8 — Candidate recall at 0.5 / 1 / 1.5 / 3 / 5 % budgets + H5 complementarity

Systems recovered by each P7 rank map vs W2, fusion, B0, B1 at the same (zone, rep, frac); `new_vs_w2` = systems P7 recovers that W2 misses at that cell. Outputs `candidate_recall/recall_<id>.csv`, `candidate_recall/h5_complementarity.csv`.

**Next decision** (predeclared, see handoff §12): T2 only if T1 is SUPPORTED or INCONCLUSIVE against T1_cnn with a non-negative worst zone, or COMPLEMENTARY; T3 only if T2 is SUPPORTED against T1 or its centerline candidate improves recall; T4 only if T2's missed systems are mostly fragmented / displaced (candidate-recall taxonomy) rather than absent.

In [ ]:
if P7_RUN['candidate_recall']:
    CR = p7.candidate_recall(EXPERIMENTS_TO_EVALUATE)
    display(CR['h5'])
    for t in CR['tables']:
        display(t)

## 9 — T4 endpoint tracer (optional; parent T2)

Per held-out discovery unit: FaultTraceGRU fitted on trajectories of the OTHER units (cross-fitted; never within 2 km of the unit, never systems withheld in it), free rollout from the endpoints of the unit's nested T2 map, bounded stops. Compared with geometric endpoint linking and the base map at the traced map's area. Output `experiments/<T2 id>/tracer/{tracer_grid.csv, tracer_history.csv, decision.json}`. Rejected if it adds lines without a pooled DTI gain ≥ 0.003 or raises FP mass by > 10 %.

In [ ]:
if P7_RUN['train_t4']:
    TRC = p7.tracer('T4')
    print(json.dumps(TRC.get('decision', TRC), indent=2, default=str))

## 10 — T5: W2 + P7 features (schema `w2_p7.v1`, optional)

Rebuilt rows (W2's sampling and seeds), matched control `w2_ctrl` (W2's frozen feature list) and `w2_p7` fitted on the same rows; held-out maps scored exactly like the frozen W2. Requires every unit's P7 held-out map (valid OOF provenance) — otherwise BLOCKED. Output `stacker/w2_p7.v1/<id>/`.

In [ ]:
if P7_RUN['w2_integration']:
    W2P7 = p7.w2_integration()
    if W2P7.get('status') == 'COMPLETE':
        display(W2P7['summary'][[c for c in ('policy', 'disc_dti', 'disc_worst', 'geo_dti', 'system_recall') if c in W2P7['summary']]])
    else:
        print(W2P7)

## 11 — Full-region inference, selection, validated export (all off by default)

* `full_inference`: zone-stitched map (each geographic fold zone from the model that held it out = the evaluated map; pixels outside every fold from the fold average, flagged) + Spearman parity per zone ≥ 0.999; centerline crests computed on the mosaic.
* `selection`: predeclared rules (disc DTI ≥ W2 + 0.005, geo ≥ 0.1758, worst-zone drop ≤ 0.010, parity, deployable policy, no leakage). Otherwise W2 stays champion; complementary rungs are listed as AUXILIARY.
* `export`: one recipe (one selection id for rep / frac / blend) through `export_parity.write_validated` into `p7_geofaultformer/submissions/`. **Never uploaded.**

In [ ]:
if P7_RUN['full_inference']:
    print(json.dumps(p7.full_inference(FULL_INFERENCE_EXPERIMENT), indent=2, default=str))
if P7_RUN['selection']:
    SEL = p7.selection()
    if 'candidates' in SEL:
        display(SEL['candidates'])
    print(json.dumps(SEL.get('decision', SEL), indent=2, default=str))
if P7_RUN['export']:
    EXP = p7.export()
    print(json.dumps({k: EXP.get(k) for k in ('status', 'reason', 'final_decision', 'export')}, indent=2, default=str))

## 12 — Figures, report, Drive synchronisation

Figures per experiment (PNG + SVG + `_data.csv`): training curves, fold DTI, probability maps, attention weights (genuine) vs input-gradient saliency (proxy), centerline comparison, orientation field, ConvGRU states, tracer trajectories, held-out examples, FP / FN maps, recall curves, discovery-vs-geographic Pareto, P7 − W2 disagreement, coverage by budget, DTI by geological domain. A figure without data is recorded ABSENT in `figure_index.json`.

In [ ]:
if P7_RUN['figures']:
    for rung in [r for r in ('T1', 'T1_cnn', 'T2', 'T3') if p7.latest_id(r)]:
        F = p7.figures(rung)
        print(rung, F)
        d = p7.exp_dir(p7.latest_id(rung)) / 'figures'
        for n in ('training_curves', 'fold_dti', 'recall_curves', 'attention_context'):
            if (d / (n + '.png')).exists():
                display(Image(filename=str(d / (n + '.png'))))
if P7_RUN['report']:
    REP = p7.report()
    display(REP['status'])
    display(Markdown(REP['markdown']))
p7.push()
print('Artifacts:', p7.ws.root, '| Drive:', Path(CONFIG['drive_root']) / p7.cfg['output_dir'])

## Execution checklist

1. GPU runtime; sections 0 → 7 with the defaults (T1 only). Read `environment/preflight.json` (blockers, memory), `folds/leakage_overlap_audit.csv`, then `comparisons/summary.csv` and `comparisons/hypotheses.csv`.
2. To test H1 at a matched trainer set `train_t1_cnn=True` and re-run 6 → 8.
3. Enable T2 / T3 / T4 one at a time, following the 'Next decision' rule of section 8.
4. T5, full inference, selection and export only after a rung is COMPLETE on every zone.
5. Disconnect: re-run 0 → 3, then the section you were in. Never tune on the public leaderboard.